# Semana 6: Funciones de Ventana y Limpieza
## Notebook de Ejercicios (NB2) – Práctica en PostgreSQL (DB-Fiddle)

### Objetivo
Poner en práctica las **funciones de ventana** (ROW_NUMBER, RANK, DENSE_RANK) y las **funciones de limpieza** de datos (texto, fechas, nulos) en un entorno SQL real utilizando **DB-Fiddle**.

### Herramienta
Utilizaremos [DB-Fiddle](https://www.db-fiddle.com/) con motor **PostgreSQL**.

### Instrucciones generales
1. Abre [DB-Fiddle](https://www.db-fiddle.com/).
2. Selecciona **PostgreSQL** como motor.
3. En el panel superior (Schema), pega el código de creación de tablas e inserción de datos.
4. En el panel inferior (Query), escribe y ejecuta las consultas propuestas.
5. Observa los resultados y reflexiona sobre cada función.

---
## 1. Creación de las Tablas de Ejemplo

Vamos a trabajar con dos tablas: **empleados** y **ventas**. La tabla empleados contiene información de empleados con sus salarios y departamentos. La tabla ventas contiene ventas diarias por tienda.

### Código SQL para crear las tablas e insertar datos

Copia y pega el siguiente código en el panel **Schema** de DB-Fiddle y ejecútalo.

In [ ]:
# Código SQL para crear las tablas (cópialo y pégalo en DB-Fiddle)
sql_create = """
-- Tabla empleados
CREATE TABLE empleados (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    departamento VARCHAR(50),
    salario DECIMAL(10,2) NOT NULL,
    fecha_contratacion DATE
);

-- Tabla ventas
CREATE TABLE ventas (
    id SERIAL PRIMARY KEY,
    tienda VARCHAR(50),
    fecha DATE NOT NULL,
    monto DECIMAL(10,2) NOT NULL
);

-- Insertar datos en empleados
INSERT INTO empleados (nombre, departamento, salario, fecha_contratacion) VALUES
    ('Juan Pérez', 'Ventas', 3500.00, '2023-01-15'),
    ('María García', 'Ventas', 4200.00, '2022-06-01'),
    ('Carlos López', 'IT', 3800.00, '2023-03-10'),
    ('Ana Martínez', 'IT', 5000.00, '2021-11-20'),
    ('Luis Fernández', 'Ventas', 3500.00, '2023-09-05'),
    ('Elena Sánchez', 'RRHH', 3200.00, '2024-01-10'),
    ('Pedro Gómez', 'IT', 4200.00, '2022-08-15'),
    ('Laura Díaz', 'Ventas', 3900.00, '2023-04-22'),
    ('Javier Ruiz', 'RRHH', 3100.00, '2024-02-14'),
    ('Sara Moreno', 'IT', 4700.00, '2022-12-01');

-- Insertar datos en ventas (con algunos formatos inconsistentes para limpieza)
INSERT INTO ventas (tienda, fecha, monto) VALUES
    ('Tienda A', '2025-01-15', 1250.00),
    ('Tienda A', '2025-01-16', 1320.00),
    ('Tienda A', '2025-01-17', 1180.00),
    ('Tienda A', '2025-01-18', 1410.00),
    ('Tienda A', '2025-01-19', 1350.00),
    ('Tienda A', '2025-01-20', 1290.00),
    ('Tienda A', '2025-01-21', 1420.00),
    ('Tienda B', '2025-01-15', 980.00),
    ('Tienda B', '2025-01-16', 1050.00),
    ('Tienda B', '2025-01-17', 1100.00),
    ('Tienda B', '2025-01-18', 1080.00),
    ('Tienda B', '2025-01-19', 1150.00),
    ('Tienda B', '2025-01-20', 1120.00),
    ('Tienda B', '2025-01-21', 1210.00),
    ('Tienda C', '2025-01-15', 750.00),
    ('Tienda C', '2025-01-16', 820.00),
    ('Tienda C', '2025-01-17', 790.00),
    ('Tienda C', '2025-01-18', 850.00),
    ('Tienda C', '2025-01-19', 880.00),
    ('Tienda C', '2025-01-20', 810.00),
    ('Tienda C', '2025-01-21', 830.00);

-- Crear una tabla con datos sucios para prácticas de limpieza
CREATE TABLE clientes_raw (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(100),
    email VARCHAR(100),
    fecha_registro VARCHAR(20),
    telefono VARCHAR(50)
);

INSERT INTO clientes_raw (nombre, email, fecha_registro, telefono) VALUES
    ('  juan perez  ', 'juan@email.com', '15/01/2025', '555-1234'),
    ('MARIA GARCIA', NULL, '16/01/2025', '555.5678'),
    ('carlos lópez', 'carlos@email', '2025-01-17', '555-9012'),
    ('ana martinez', 'ana@email.com', '18/01/2025', '555 3456'),
    ('luis fernandez', 'luis@email.com', '19-01-2025', NULL);
"""
print("✅ Código SQL listo para copiar a DB-Fiddle.")

---
## 2. Ejercicios con Funciones de Ranking

### Ejercicio 2.1: ROW_NUMBER()

Asigna un número de fila único a cada empleado dentro de su departamento, ordenados por salario descendente.

**SQL esperado:**

In [ ]:
sql_ej2_1 = """
SELECT 
    nombre,
    departamento,
    salario,
    ROW_NUMBER() OVER (PARTITION BY departamento ORDER BY salario DESC) AS row_num
FROM empleados
ORDER BY departamento, salario DESC;
"""
print("✅ Consulta 2.1 lista.")

### Ejercicio 2.2: RANK() y DENSE_RANK()

Observa la diferencia entre RANK y DENSE_RANK cuando hay salarios iguales. Añade un empleado con salario duplicado para ver el efecto.

**Primero, inserta un empleado duplicado (ejecuta en Query):**

```sql
INSERT INTO empleados (nombre, departamento, salario) VALUES ('Pedro Prueba', 'Ventas', 3500.00);
```

Luego ejecuta:

In [ ]:
sql_ej2_2 = """
SELECT 
    nombre,
    departamento,
    salario,
    RANK() OVER (PARTITION BY departamento ORDER BY salario DESC) AS rank,
    DENSE_RANK() OVER (PARTITION BY departamento ORDER BY salario DESC) AS dense_rank
FROM empleados
ORDER BY departamento, salario DESC;
"""
print("✅ Consulta 2.2 lista.")

### Ejercicio 2.3: Top N por departamento

Obtén los 2 empleados mejor pagados de cada departamento.

**SQL esperado:**

In [ ]:
sql_ej2_3 = """
WITH ranked AS (
    SELECT 
        nombre,
        departamento,
        salario,
        ROW_NUMBER() OVER (PARTITION BY departamento ORDER BY salario DESC) AS rn
    FROM empleados
)
SELECT nombre, departamento, salario
FROM ranked
WHERE rn <= 2
ORDER BY departamento, salario DESC;
"""
print("✅ Consulta 2.3 lista.")

---
## 3. Funciones de Ventana: LAG y LEAD

### Ejercicio 3.1: Diferencia con el día anterior

Para la Tienda A, calcula las ventas del día anterior y la diferencia con el día actual.

**SQL esperado:**

In [ ]:
sql_ej3_1 = """
SELECT 
    fecha,
    monto,
    LAG(monto, 1) OVER (ORDER BY fecha) AS monto_anterior,
    monto - LAG(monto, 1) OVER (ORDER BY fecha) AS diferencia
FROM ventas
WHERE tienda = 'Tienda A'
ORDER BY fecha;
"""
print("✅ Consulta 3.1 lista.")

### Ejercicio 3.2: Media móvil de 3 días

Calcula la media móvil de 3 días para las ventas de la Tienda B.

**SQL esperado:**

In [ ]:
sql_ej3_2 = """
SELECT 
    fecha,
    monto,
    AVG(monto) OVER (ORDER BY fecha ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS media_movil_3
FROM ventas
WHERE tienda = 'Tienda B'
ORDER BY fecha;
"""
print("✅ Consulta 3.2 lista.")

---
## 4. Limpieza de Datos con Funciones de Texto

Trabajaremos con la tabla `clientes_raw` que contiene datos sucios.

### Ejercicio 4.1: Limpiar nombres

Elimina espacios extras y convierte a formato título (primera letra mayúscula).

**SQL esperado:**

In [ ]:
sql_ej4_1 = """
SELECT 
    id,
    TRIM(nombre) AS nombre_sin_espacios,
    INITCAP(TRIM(nombre)) AS nombre_limpio
FROM clientes_raw;
"""
print("✅ Consulta 4.1 lista.")

### Ejercicio 4.2: Manejo de emails nulos

Reemplaza los emails nulos por un valor por defecto 'pendiente@mail.com'.

**SQL esperado:**

In [ ]:
sql_ej4_2 = """
SELECT 
    id,
    TRIM(nombre) AS nombre,
    COALESCE(email, 'pendiente@mail.com') AS email_limpio
FROM clientes_raw;
"""
print("✅ Consulta 4.2 lista.")

### Ejercicio 4.3: Validar formato de email

Identifica los emails que no contienen '@' (considerados inválidos). Usa `NOT LIKE` o expresiones regulares.

**SQL esperado:**

In [ ]:
sql_ej4_3 = """
SELECT 
    id,
    nombre,
    email,
    CASE 
        WHEN email IS NULL THEN 'Sin email'
        WHEN email NOT LIKE '%@%' THEN 'Email inválido'
        ELSE 'Válido'
    END AS estado_email
FROM clientes_raw;
"""
print("✅ Consulta 4.3 lista.")

---
## 5. Limpieza de Fechas

La columna `fecha_registro` en `clientes_raw` tiene formatos inconsistentes. Vamos a convertirla a tipo DATE.

### Ejercicio 5.1: Convertir a DATE

Usa `TO_DATE` con diferentes formatos según el caso. Esto requiere lógica condicional.

**SQL esperado:**

In [ ]:
sql_ej5_1 = """
SELECT 
    id,
    nombre,
    fecha_registro,
    CASE 
        WHEN fecha_registro LIKE '__/__/____' THEN TO_DATE(fecha_registro, 'DD/MM/YYYY')
        WHEN fecha_registro LIKE '____-__-__' THEN TO_DATE(fecha_registro, 'YYYY-MM-DD')
        WHEN fecha_registro LIKE '__-__-____' THEN TO_DATE(fecha_registro, 'DD-MM-YYYY')
        ELSE NULL
    END AS fecha_convertida
FROM clientes_raw;
"""
print("✅ Consulta 5.1 lista.")

### Ejercicio 5.2: Extraer componentes de fecha

Para las fechas convertidas, extrae el año y el mes.

**SQL esperado (usando la consulta anterior como subconsulta o CTE):**

In [ ]:
sql_ej5_2 = """
WITH fechas_limpias AS (
    SELECT 
        id,
        nombre,
        CASE 
            WHEN fecha_registro LIKE '__/__/____' THEN TO_DATE(fecha_registro, 'DD/MM/YYYY')
            WHEN fecha_registro LIKE '____-__-__' THEN TO_DATE(fecha_registro, 'YYYY-MM-DD')
            WHEN fecha_registro LIKE '__-__-____' THEN TO_DATE(fecha_registro, 'DD-MM-YYYY')
            ELSE NULL
        END AS fecha
    FROM clientes_raw
)
SELECT 
    id,
    nombre,
    fecha,
    EXTRACT(YEAR FROM fecha) AS año,
    EXTRACT(MONTH FROM fecha) AS mes
FROM fechas_limpias;
"""
print("✅ Consulta 5.2 lista.")

---
## 6. Limpieza de Teléfonos

### Ejercicio 6.1: Normalizar números de teléfono

Los teléfonos tienen separadores inconsistentes ('.', '-', ' '). Limpia para que todos tengan formato '555-1234'.

**SQL esperado (usa `REGEXP_REPLACE` o `REPLACE` encadenado):**

In [ ]:
sql_ej6_1 = """
SELECT 
    id,
    telefono,
    REGEXP_REPLACE(telefono, '[^0-9]', '', 'g') AS solo_numeros,
    CASE 
        WHEN telefono IS NULL THEN NULL
        ELSE SUBSTRING(REGEXP_REPLACE(telefono, '[^0-9]', '', 'g') FROM 1 FOR 3) || '-' || 
             SUBSTRING(REGEXP_REPLACE(telefono, '[^0-9]', '', 'g') FROM 4 FOR 4)
    END AS telefono_normalizado
FROM clientes_raw;
"""
print("✅ Consulta 6.1 lista.")

---
## 7. Ejercicio Integrador

Combina varias técnicas: obtén un listado limpio de clientes con nombre en formato título, email válido (o 'pendiente@mail.com'), fecha de registro como DATE, y teléfono normalizado.

**SQL esperado:**

In [ ]:
sql_integrador = """
WITH clientes_con_fecha AS (
    SELECT 
        id,
        INITCAP(TRIM(nombre)) AS nombre_limpio,
        COALESCE(
            CASE WHEN email LIKE '%@%' THEN email ELSE NULL END, 
            'pendiente@mail.com'
        ) AS email_limpio,
        CASE 
            WHEN fecha_registro LIKE '__/__/____' THEN TO_DATE(fecha_registro, 'DD/MM/YYYY')
            WHEN fecha_registro LIKE '____-__-__' THEN TO_DATE(fecha_registro, 'YYYY-MM-DD')
            WHEN fecha_registro LIKE '__-__-____' THEN TO_DATE(fecha_registro, 'DD-MM-YYYY')
            ELSE NULL
        END AS fecha_limpia,
        REGEXP_REPLACE(telefono, '[^0-9]', '', 'g') AS solo_numeros
    FROM clientes_raw
)
SELECT 
    id,
    nombre_limpio,
    email_limpio,
    fecha_limpia,
    CASE 
        WHEN solo_numeros IS NULL OR solo_numeros = '' THEN NULL
        ELSE SUBSTRING(solo_numeros FROM 1 FOR 3) || '-' || SUBSTRING(solo_numeros FROM 4 FOR 4)
    END AS telefono_normalizado
FROM clientes_con_fecha;
"""
print("✅ Consulta integradora lista.")

---
## 8. Reflexión y Análisis

Después de ejecutar las consultas, reflexiona sobre las siguientes preguntas:

1. ¿Cuál es la diferencia práctica entre ROW_NUMBER, RANK y DENSE_RANK?
2. ¿Por qué es importante limpiar los datos antes de usarlos en análisis o modelos de IA?
3. ¿Qué desafíos encuentras al trabajar con formatos de fecha inconsistentes?
4. ¿Cómo afectan los valores nulos a las funciones de ventana?

Escribe tus respuestas en una celda de texto.

In [ ]:
# Escribe aquí tus reflexiones
print("""
1. ROW_NUMBER asigna números únicos; RANK deja huecos en caso de empates; DENSE_RANK no deja huecos.
2. Datos sucios llevan a análisis incorrectos y modelos de IA sesgados o imprecisos.
3. Fechas inconsistentes requieren lógica condicional y pueden causar errores de conversión.
4. Los nulos pueden excluirse o manejarse con COALESCE; algunas funciones de ventana los ignoran.
""")

---
## Ejercicios para el Estudiante (Tarea)

1.  **Añade más datos:** Inserta nuevos registros en `clientes_raw` con otros formatos de fecha y teléfono, y repite las consultas de limpieza.

2.  **Crea tus propias consultas:** Inventa al menos 2 consultas adicionales que utilicen funciones de ventana (por ejemplo, percentiles con `NTILE`).

3.  **Analiza el rendimiento:** Para una de las consultas con funciones de ventana, usa `EXPLAIN` (anteponiendo EXPLAIN a la consulta) y analiza el plan de ejecución.

4.  **Traduce a pandas:** Elige una de las consultas de limpieza y escribe el código Python con pandas que realizaría la misma operación (similar al NB1).

5.  **Reflexión final:** ¿En qué situaciones del mundo real usarías cada función de ventana y limpieza?

---
## Conclusiones de la Sesión

*   Hemos practicado las **funciones de ventana** (`ROW_NUMBER`, `RANK`, `DENSE_RANK`) en PostgreSQL.
*   Implementamos **LAG** y **media móvil** para análisis de series temporales.
*   Aplicamos **funciones de limpieza** de texto (`TRIM`, `INITCAP`, `COALESCE`, expresiones regulares).
*   Limpiamos fechas inconsistentes usando `TO_DATE` con lógica condicional.
*   Normalizamos números de teléfono eliminando caracteres no numéricos.
*   Integramos varias técnicas en una consulta final de limpieza completa.
*   Estas habilidades son fundamentales en procesos ETL y preparación de datos para análisis y machine learning.

¡Ahora dominas las funciones de ventana y limpieza en SQL!